<a href="https://colab.research.google.com/github/anokhina-rgb/Consecutive-Translation-Trainer/blob/main/Google_Colab_Single_Run_Multi_Lang_Version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # Lingua Wave Studio - Multi-Language All-in-One Run 🌊🎙_
#
# Налаштуйте параметри праворуч, запустіть комірку, завантажте ваш MP3-файл, і скрипт автоматично:
# 1. Встановить усі бібліотеки.
# 2. Обробить ваш завантажений аудіофайл.
# 3. Розпізнає оригінальний текст (Whisper).
# 4. Перекладе його на **всі вказані вами мови** одночасно.
# 5. Для кожної мови згенерує окремий файл TTS (.mp3) з паузами та документ Word (.docx).
# 6. Скачає весь пакет результатів у ваш браузер.

# %%
#@title ⚙️ Налаштування та Запуск { run: "auto" }
src_lang = "en" #@param ["en", "uk", "fr", "de", "es", "sk"]
# Вкажіть коди мов перекладу через кому (наприклад: uk, es, sk)
tgt_langs_input = "de" #@param {type:"string"}
pause_ms = 2000 #@param {type:"integer"}

import os
import sys
import time
from datetime import datetime

# --- КРОК 1: Автоматичне встановлення залежностей ---
print("⏳ Перевірка та встановлення необхідних системних утиліт і бібліотек...")
if os.system("ffmpeg -version > /dev/null 2>&1") != 0:
    print("📦 Встановлення FFmpeg...")
    os.system("apt-get install -y ffmpeg > /dev/null 2>&1")

try:
    import faster_whisper
    from deep_translator import GoogleTranslator
    from gtts import gTTS
    from pydub import AudioSegment
    from docx import Document
except ImportError:
    print("📦 Встановлення пакетів Python (faster-whisper, deep-translator, gTTS, pydub, python-docx)...")
    os.system("pip install -q faster-whisper deep-translator gTTS pydub python-docx pandas")
    print("✅ Бібліотеки встановлено!")

# Імпортуємо модулі
from google.colab import files
from faster_whisper import WhisperModel
from deep_translator import GoogleTranslator
from gtts import gTTS
from pydub import AudioSegment
from docx import Document

# Парсимо мови перекладу
tgt_langs = [lang.strip().lower() for lang in tgt_langs_input.split(",") if lang.strip()]

# --- КРОК 2: Завантаження файлу користувачем ---
print("\n" + "="*50)
print("📥 КРОК 2: ВИБЕРІТЬ ВАШ MP3-ФАЙЛ З КОМП'ЮТЕРА")
print("="*50)
uploaded = files.upload()

if not uploaded:
    print("❌ Помилка: Файл не вибрано. Запустіть комірку знову.")
else:
    uploaded_audio_path = list(uploaded.keys())[0]
    print(f"✅ Файл успішно завантажено: {uploaded_audio_path}")
    print(f"🔄 Параметри: Оригінал ({src_lang}) ➔ Переклад на мови: {tgt_langs} | Пауза: {pause_ms} мс")

    # --- КРОК 3: Транскрипція оригіналу (один раз для економії часу) ---
    print("\n" + "="*50)
    print("🎙️ КРОК 3: РОЗПІЗНАВАННЯ ТЕКСТУ (Whisper)")
    print("="*50)
    print("⏳ Завантаження моделі Whisper...")

    # Використовуємо CPU з типом int8 для швидкої роботи в безкоштовному Colab
    model = WhisperModel("small", device="cpu", compute_type="int8")

    print("🎤 Розпізнавання оригінальної мови в аудіо...")
    segments, _ = model.transcribe(uploaded_audio_path)

    original_segments = []
    for segment in segments:
        text = segment.text.strip()
        if text:
            original_segments.append(text)
            print(f"[{time.strftime('%H:%M:%S')}] {text}")

    if not original_segments:
        print("⚠️ Не вдалося розпізнати жодного тексту в наданому аудіофайлі.")
    else:
        print(f"\n✅ Успішно розпізнано {len(original_segments)} фрагментів тексту.")

        # --- КРОК 4: Переклад та генерація TTS для КОЖНОЇ мови ---
        print("\n" + "="*50)
        print("🔊 КРОК 4: ПЕРЕКЛАД, ГЕНЕРАЦІЯ TTS ТА СТВОРЕННЯ ФАЙЛІВ")
        print("="*50)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        generated_files = []

        for lang in tgt_langs:
            print(f"\n🌍 Обробка мови перекладу: [{lang.upper()}]")
            try:
                translator = GoogleTranslator(source=src_lang, target=lang)
            except Exception as e:
                print(f"❌ Помилка ініціалізації перекладача для мови {lang}: {e}")
                continue

            combined = AudioSegment.silent(duration=0)
            doc = Document()

            # Перекладаємо та озвучуємо кожен фрагмент
            for i, orig_text in enumerate(original_segments):
                try:
                    # Переклад
                    translated_text = translator.translate(orig_text)
                    print(f"  [{i+1}/{len(original_segments)}] {orig_text} ➔ {translated_text}")

                    # Запис у Word документ
                    doc.add_paragraph(translated_text)

                    # Генерація аудіо (TTS)
                    tts = gTTS(text=translated_text, lang=lang)
                    temp_mp3 = f"temp_{lang}_{i}.mp3"
                    tts.save(temp_mp3)

                    # Склеювання аудіо з паузою
                    combined += AudioSegment.from_file(temp_mp3) + AudioSegment.silent(duration=pause_ms)

                    # Очищення тимчасового файлу
                    if os.path.exists(temp_mp3):
                        os.remove(temp_mp3)
                except Exception as e:
                    print(f"  ⚠️ Помилка обробки фрагмента {i+1} для мови {lang}: {e}")

            # Збереження готових результатів для поточної мови
            mp3_filename = f"result_{lang}_{timestamp}.mp3"
            docx_filename = f"result_{lang}_{timestamp}.docx"

            combined.export(mp3_filename, format="mp3")
            doc.save(docx_filename)

            generated_files.append(mp3_filename)
            generated_files.append(docx_filename)
            print(f"✅ Готово для [{lang.upper()}]: створено {mp3_filename} та {docx_filename}")

        # Видаляємо завантажений користувачем оригінал для економії місця
        if os.path.exists(uploaded_audio_path):
            os.remove(uploaded_audio_path)

        # --- КРОК 5: Автоматичне завантаження всього створеного ---
        print("\n" + "="*50)
        print("💾 КРОК 5: АВТОМАТИЧНЕ СКАЧУВАННЯ ФАЙЛІВ")
        print("="*50)
        print("Зачекайте, завантажуємо ваші файли...")

        for file_to_download in generated_files:
            if os.path.exists(file_to_download):
                files.download(file_to_download)

        print("\n🎉 Все готово! Перевірте теку 'Завантаження' (Downloads).")

⏳ Перевірка та встановлення необхідних системних утиліт і бібліотек...
📦 Встановлення пакетів Python (faster-whisper, deep-translator, gTTS, pydub, python-docx)...
